# Ноутбук 2 — гружу модель и проверяю, что она работает

Беру базовую модель `deepvk/llava-gemma-2b-lora`, гружу в fp16 (3B спокойно влезает в T4)
и проверяю инференс: даю картинку и вопрос, смотрю ответ. В конце смотрю, сколько заняло видеопамяти.

## Ставлю библиотеки

Модель от 2024 года, а в Colab сейчас очень свежий torch и transformers — на них модель либо
падает при импорте, либо выдаёт мусор вместо описания. Поэтому воссоздаю окружение под модель:
старый torch и transformers уровня 2024 года. Pillow тоже ниже 12, иначе ломается импорт.

**После установки обязательно перезапусти сессию** (Среда выполнения -> Перезапустить сеанс),
иначе новый torch, который загружен в Colab по умолчанию, не сменится на нужный.

In [ ]:
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "huggingface_hub==0.24.6" "accelerate==0.33.0" "pillow<12"

После перезапуска сессии проверяю, что версии встали правильно.

In [ ]:
import torch, transformers
print(torch.__version__, transformers.__version__)
print('GPU:', torch.cuda.is_available())

## Логинюсь в HuggingFace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Гружу модель в fp16

In [ ]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, AutoTokenizer

model_id = 'deepvk/llava-gemma-2b-lora'

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

print('модель загрузилась')
print(model.config.model_type)

## Пробую инференс на картинке из примера

Беру ту же картинку со стоп-знаком, что в карточке модели, и прошу описать её.

In [ ]:
import requests
from PIL import Image

url = 'https://www.ilankelman.org/stopsigns/australia.jpg'
img = Image.open(requests.get(url, stream=True).raw).convert('RGB')
img

In [ ]:
messages = [
    {'role': 'user', 'content': '<image>\nОпиши картинку несколькими словами.'}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(images=[img], text=text, return_tensors='pt').to(model.device)
inputs['pixel_values'] = inputs['pixel_values'].to(torch.float16)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30)

answer = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(answer)

## Смотрю, сколько заняло видеопамяти

In [ ]:
used = torch.cuda.memory_allocated() / 1e9
peak = torch.cuda.max_memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'сейчас занято : {used:.2f} GB')
print(f'пик          : {peak:.2f} GB')
print(f'всего на GPU : {total:.2f} GB')
print(f'свободно     : {total - peak:.2f} GB')

## Что получилось

Модель грузится в fp16 и на картинку выдаёт нормальное описание — базовая часть работает.
Проверял на стоп-знаке: получил 'На улице стоит красный стоп-знак, рядом с которым находится автомобиль.'

Ключевой момент: пришлось воссоздать окружение 2024 года (torch 2.4.1 + transformers 4.44.2),
иначе на свежих версиях Colab модель либо не импортируется, либо выдаёт бессмысленный текст
(несовместимость чекпоинта 2024 года с новым кодом torch/LLaVA). Этот же набор версий использую
дальше для оценки и обучения.

Дальше: прогнать эту модель без обучения на GQA-ru и MMBench-ru и записать метрики — это точка отсчёта.